# PA1 on Kaggle

Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**. Add a PACS dataset under *Add Input* (folders for photo / art_painting / cartoon / sketch).

This notebook only runs the `.py` scripts. All code lives in the repo.

In [ ]:
# Get the code. Pick ONE option.
# A) after the repo is on GitHub:
# !git clone https://github.com/<user>/<repo>.git /kaggle/working/pa1
# B) upload the pa1 folder as a private Kaggle Dataset, then:
# !cp -r /kaggle/input/<your-code-dataset> /kaggle/working/pa1
%cd /kaggle/working/pa1

In [ ]:
# Kaggle already has torch, torchvision, sklearn, pandas, matplotlib.
!pip install -q open_clip_torch umap-learn
import os
os.environ["PA1_DATA_ROOT"] = "/kaggle/working/data"
os.environ["PA1_OUT_ROOT"] = "/kaggle/working"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# PACS: link the input dataset (read-only) and make the source splits once.
!mkdir -p /kaggle/working/data
!ln -sfn /kaggle/input/<pacs-dataset> /kaggle/working/data/pacs
!python -m shared.make_pacs_splits

In [ ]:
# Task 2: two runs at a time, one per GPU. Kaggle has 4 CPUs, so keep loader workers low.
!bash kaggle/run_parallel.sh task2.train task2/configs/source_only.yaml task2/configs/dan.yaml train.num_workers_per_loader=1
!bash kaggle/run_parallel.sh task2.train task2/configs/dann.yaml task2/configs/cdan.yaml train.num_workers_per_loader=1
!tail -n 3 logs/*.log

In [ ]:
# Task 3 (needs task2 source_only checkpoint for ERM).
!bash kaggle/run_parallel.sh task3.train task3/configs/dan_dg.yaml task3/configs/sam.yaml train.num_workers_per_loader=1

In [ ]:
# Task 4 preflight (~3 min): every Task 4 step for a few batches on this GPU, in a temp folder.
# It also downloads CIFAR-10/100 once. Do NOT start the real run unless the last line says PREFLIGHT OK.
!bash task4/preflight.sh

In [ ]:
# Task 4: Vanilla and GCSC at the same time, one per GPU (~1 h). Logs: logs/vanilla.log, logs/gcsc.log
# If this cell stops for any reason, just run it again: each run resumes from its last finished epoch.
!bash kaggle/run_parallel.sh task4.train task4/configs/vanilla.yaml task4/configs/gcsc.yaml train.num_workers=2
!tail -n 2 logs/vanilla.log logs/gcsc.log

In [ ]:
# Task 4: PROSER starts from the vanilla checkpoint (50 epochs). Also resumes if rerun. Log: logs/proser.log
!python -m task4.train --config task4/configs/proser.yaml train.num_workers=2 > logs/proser.log 2>&1; tail -n 3 logs/proser.log

In [ ]:
# Task 4: ONLY after all three models are trained: save outputs (incl. CIFAR-100 unknowns), then evaluate.
!for r in vanilla gcsc proser; do python -m task4.extract_outputs --run $r --include-unknowns; done
!python -m task4.evaluate_osr
# Download these before the session ends: task4/results/ (small) and checkpoints/task4/ (optional).
!cd /kaggle/working && zip -qr task4_out.zip pa1/task4/results pa1/task4/cache checkpoints/task4 && ls -lh task4_out.zip

In [ ]:
# Save results + checkpoints before the session ends (Output tab keeps /kaggle/working).
!ls -R /kaggle/working/checkpoints | head -50